# DocTrack — Banc d'essai de la transcription (wolof + français)

**But :** mesurer, sur de vrais vocaux, quel modèle gratuit comprend le mieux les Sénégalais qui parlent à DocTrack.

**Avant de commencer**
1. Menu **Exécution → Modifier le type d'exécution → GPU T4** (gratuit).
2. Préparez un fichier **`corpus.zip`** contenant :
   - les vocaux (`S03_P2_wolof.ogg`, `S01_P1_mixte.m4a`…) ;
   - un fichier **`transcriptions.csv`** (séparateur `,`) avec les colonnes `fichier,langue,texte` :
     `S03_P2_wolof.ogg,wo,dama gis ab kart didantite ca marse ba` — le **texte exact** de ce qui est dit (c'est la « bonne réponse »).
     `langue` = `wo`, `fr` ou `mixte`.
3. Exécutez les cellules une par une (▶). Comptez 10 à 20 minutes au total.

**Ce que mesure le test**
- **WER** (taux d'erreur sur les mots) et **CER** (sur les lettres) : plus c'est bas, mieux c'est. En wolof, le CER est plus parlant (l'orthographe varie beaucoup).
- **Mots-clés DocTrack retrouvés** : proportion des mots utiles (*réer, gis, kart, paspoor, perdu, trouvé…*) bien reconnus. C'est ce qui compte pour l'assistant, qui comprend par mots-clés.
- **Temps** de transcription par vocal.

⚠️ Données : uniquement des vocaux de personnes d'accord, avec des noms et numéros **inventés**.

## 1. Installation (5 à 8 minutes)

In [ ]:
!pip -q install omnilingual-asr==0.2.0 jiwer pandas matplotlib
!apt-get -qq install -y ffmpeg > /dev/null
import torch; print('GPU :', torch.cuda.is_available(), '| torch', torch.__version__)

## 2. Envoyer `corpus.zip`

In [ ]:
from google.colab import files
import zipfile, os, shutil
shutil.rmtree('corpus', ignore_errors=True)
up = files.upload()                      # choisissez corpus.zip
name = next(iter(up))
zipfile.ZipFile(name).extractall('corpus')
# le CSV peut être dans un sous-dossier
csv_path = next(os.path.join(r, f) for r, _, fs in os.walk('corpus') for f in fs if f.lower() == 'transcriptions.csv')
base = os.path.dirname(csv_path)
import pandas as pd
ref = pd.read_csv(csv_path)
ref.columns = [c.strip().lower() for c in ref.columns]
ref['chemin'] = ref['fichier'].map(lambda f: os.path.join(base, str(f).strip()))
manquants = ref[~ref['chemin'].map(os.path.exists)]
print(len(ref), 'vocaux —', len(manquants), 'fichier(s) introuvable(s)')
display(manquants[['fichier']]) if len(manquants) else None
ref = ref[ref['chemin'].map(os.path.exists)].reset_index(drop=True)
ref.groupby('langue').size()

## 3. Conversion en WAV 16 kHz mono (format attendu par les modèles)

In [ ]:
import subprocess
os.makedirs('wav', exist_ok=True)
def to_wav(src):
    dst = os.path.join('wav', os.path.splitext(os.path.basename(src))[0] + '.wav')
    subprocess.run(['ffmpeg','-nostdin','-loglevel','error','-y','-i',src,'-t','39','-ac','1','-ar','16000',dst], check=True)
    return dst
ref['wav'] = ref['chemin'].map(to_wav)
dur = lambda p: float(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','csv=p=0',p],capture_output=True,text=True).stdout or 0)
ref['duree_s'] = ref['wav'].map(dur)
print('Durée totale :', round(ref['duree_s'].sum()/60, 1), 'min')

## 4. Transcription par chaque modèle

Modèles testés (tous gratuits, licence Apache 2.0) :
- `omniASR_CTC_300M_v2` — petit et rapide : celui qui tourne sur le Space gratuit.
- `omniASR_CTC_1B_v2` — plus gros, souvent plus précis.
- `omniASR_LLM_1B_v2` — utilise la langue indiquée (`wol_Latn` / `fra_Latn`).

Vous pouvez retirer un modèle de la liste pour aller plus vite.

In [ ]:
import time, gc
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
MODELES = ['omniASR_CTC_300M_v2', 'omniASR_CTC_1B_v2', 'omniASR_LLM_1B_v2']
CODE = {'wo': 'wol_Latn', 'fr': 'fra_Latn', 'mixte': 'wol_Latn'}
resultats = []
for m in MODELES:
    print('▶', m)
    pipe = ASRInferencePipeline(model_card=m)
    for _, row in ref.iterrows():
        t = time.time()
        try:
            hyp = pipe.transcribe([row['wav']], lang=[CODE.get(str(row['langue']).strip(), 'wol_Latn')], batch_size=1)[0]
        except Exception as e:
            hyp = ''; print('  erreur', row['fichier'], e)
        resultats.append({'modele': m, 'fichier': row['fichier'], 'langue': row['langue'],
                          'reference': row['texte'], 'transcription': hyp, 'temps_s': round(time.time()-t, 2), 'duree_s': row['duree_s']})
    del pipe; gc.collect(); torch.cuda.empty_cache()
res = pd.DataFrame(resultats)
res.head(10)

## 5. Scores

In [ ]:
import re, unicodedata, jiwer
def norm(s):
    s = unicodedata.normalize('NFKD', str(s).lower())
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"[^a-z0-9 ]", ' ', s.replace("'", ' '))
    return re.sub(r'\s+', ' ', s).strip()

# Mots utiles à l'assistant (mêmes familles que config/assistant.php)
MOTS_CLES = ['reer','rer','nakk','sacc','gis','fekk','for','perdu','perte','trouve','kart','carte','identite','cni','paspoor','passeport',
             'permis','grise','naissance','fan','ana','indi','waaw','deedeet','deet','xamuma','oui','non','dossier','agent','recuperer',
             'dakar','pikine','ziguinchor','kolda','thies','marse','marche','taxi']
def mots_cles(ref_, hyp_):
    r = set(norm(ref_).split()) & set(MOTS_CLES)
    if not r: return None
    return len(r & set(norm(hyp_).split())) / len(r)

res['wer'] = [jiwer.wer(norm(r) or '-', norm(h) or '-') for r, h in zip(res['reference'], res['transcription'])]
res['cer'] = [jiwer.cer(norm(r) or '-', norm(h) or '-') for r, h in zip(res['reference'], res['transcription'])]
res['mots_cles'] = [mots_cles(r, h) for r, h in zip(res['reference'], res['transcription'])]
synthese = res.groupby(['modele','langue']).agg(vocaux=('fichier','count'), WER=('wer','mean'), CER=('cer','mean'),
                                               mots_cles=('mots_cles','mean'), temps_moyen_s=('temps_s','mean')).round(3)
globale = res.groupby('modele').agg(WER=('wer','mean'), CER=('cer','mean'), mots_cles=('mots_cles','mean'), temps_moyen_s=('temps_s','mean')).round(3)
display(globale.sort_values('CER'))
synthese

In [ ]:
import matplotlib.pyplot as plt
ax = globale[['CER','WER']].plot.bar(figsize=(8,4), rot=0, title="Taux d'erreur par modèle (plus bas = mieux)")
ax.set_ylabel('taux'); plt.tight_layout(); plt.show()
(globale['mots_cles']*100).plot.bar(figsize=(8,3), rot=0, title='Mots-clés DocTrack retrouvés (%)', color='seagreen'); plt.tight_layout(); plt.show()

## 6. Les pires erreurs (pour comprendre ce qui coince)

In [ ]:
pd.set_option('display.max_colwidth', 120)
meilleur = globale['CER'].idxmin()
print('Meilleur modèle :', meilleur)
res[res.modele == meilleur].sort_values('cer', ascending=False)[['fichier','langue','reference','transcription','cer']].head(10)

## 7. Télécharger les résultats

Envoyez `resultats_transcription.csv` et `synthese_transcription.csv` à Claude dans le projet : ils serviront au workplan (chiffres réels) et à améliorer les mots-clés de l'assistant.

In [ ]:
res.to_csv('resultats_transcription.csv', index=False)
synthese.to_csv('synthese_transcription.csv')
files.download('resultats_transcription.csv'); files.download('synthese_transcription.csv')